In [1]:
##imports
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [5]:
##Load Dataset
import pandas as pd
DATASET_PATH='dataset/'
csv_path=DATASET_PATH+"test_dataset.csv"

df=pd.read_csv(csv_path)


df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 150 non-null    int64 
 1   Title              150 non-null    object
 2   Poet               150 non-null    object
 3   text               150 non-null    object
 4   ctext              150 non-null    object
 5   Poem Link          150 non-null    object
 6   our_summary        150 non-null    object
 7   prompt_by_summary  150 non-null    object
 8   video_path         12 non-null     object
dtypes: int64(1), object(8)
memory usage: 10.7+ KB


,id,Title,Poet,text,ctext,Poem Link,our_summary,prompt_by_summary,video_path
0,0,"Dear John, Dear Coltrane by Michael S. Harper",Michael S. Harper,"'Dear John, Dear Coltrane' by Michael S. Harpe...","a love supreme, a love supreme\na love supreme...",https://www.poetryfoundation.org/poems/42827/d...,"The poem explores themes of love, loss, pain, ...",A lone musician stands on a dimly lit stage un...,videos/0.mp4
1,1,Parrot by Stevie Smith,Stevie Smith,‘Parrot‘ depicts the declining health of a won...,The old sick green parrot\nHigh in a dingy cag...,https://revise.wales/pastPapers/A-level/Englis...,"This old parrot, sick and full of rage, longs ...",A weathered and aged parrot with vibrant yet m...,videos/1.mp4
2,2,Dust of Snow by Robert Frost,Robert Frost,"The simplicity, in the end, is the key element...",The way a crow\nShook down on me\nThe dust of ...,https://www.poetryfoundation.org/poems/44262/d...,The sight of a crow shaking snow from a tree t...,"A solitary poet, bundled in a woolen coat and ...",videos/2.mp4
3,3,Suburban Sonnet by Gwen Harwood,Gwen Harwood,'Suburban Sonnet' by Gwen Harwood is a poem ab...,"She practises a fugue, though it can matter\nt...",https://genius.com/Gwen-harwood-suburban-sonne...,"A mother practices music, but her children int...",A mother sits at a grand piano in a cozy livin...,videos/3.mp4
4,4,Unending Love by Rabindranath Tagore,Rabindranath Tagore,'Unending Love' by Rabindranath Tagore is a he...,"I seem to have loved you in numberless forms, ...",https://allpoetry.com/Unending-Love,The speaker expresses their eternal love for s...,A person stands in a serene open field under a...,videos/4.mp4


In [3]:
model_path = '../models/VPO-5B'  # replace with your model path
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

model = AutoModelForCausalLM.from_pretrained(model_path).half().eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# ------------------------------
# 1. Prompt Template
# ------------------------------
prompt_template = """In this task, your goal is to expand the user's short query into a detailed and well-structured English prompt for generating short videos.

Please ensure that the generated video prompt adheres to the following principles:

1. **Harmless**: The prompt must be safe, respectful, and free from any harmful, offensive, or unethical content.  
2. **Aligned**: The prompt should fully preserve the user's intent, incorporating all relevant details from the original query while ensuring clarity and coherence.  
3. **Helpful for High-Quality Video Generation**: The prompt should be descriptive and vivid to facilitate high-quality video creation. Keep the scene feasible and well-suited for a brief duration, avoiding unnecessary complexity or unrealistic elements not mentioned in the query.

User Query: {}

Video Prompt:
"""

# ------------------------------
# 2. Function to Generate Video Prompt
# ------------------------------
def generate_video_prompt(text):
    message = [{'role': 'user', 'content': prompt_template.format(text)}]

    # ✅ Build prompt string
    prompt = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=False
    )

    # ✅ Tokenize into a dictionary (correct input format)
    model_inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    # ✅ Generate
    with torch.no_grad():
        output = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            do_sample=True,
            top_p=1.0,
            temperature=0.7,
            num_beams=1
        )

    # ✅ Decode clean text
    decoded = tokenizer.decode(output[0])

    # ✅ Extract only assistant response
    if "<|start_header_id|>assistant<|end_header_id|>" in decoded:
        decoded = decoded.split("<|start_header_id|>assistant<|end_header_id|>", 1)[1]

    if "<|eot_id|>" in decoded:
        decoded = decoded.split("<|eot_id|>", 1)[0]

    return decoded.strip()



In [5]:
print(df.iloc[1]['Poem Link'])

print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

print(df.iloc[1]['our_summary'])

print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
generate_video_prompt(df.iloc[1]['our_summary'])

# generate_video_prompt(df.iloc[0]['our_summary'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


https://revise.wales/pastPapers/A-level/EnglishLit/unit3/s17-2727-01.pdf
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
This old parrot, sick and full of rage, longs for his jungle home but suffers in his new, urban surroundings. He is plagued by illness and waits for death to relieve him of his misery.
+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


'A weathered, elderly parrot with vibrant green and blue feathers sits on a small perch inside a dimly lit urban apartment. Its eyes, though sunken and weary, still carry a deep longing that contrasts with its otherwise frail and emaciated appearance. The surrounding environment is sparse, with a view of concrete walls and distant urban lights through a small window. The parrot occasionally tilts its head, gazing at a framed photograph of lush greenery on the wall, reminiscent of its former jungle home. Its movements are slow and labored, hinting at its illness, as it softly tilts its head in melancholy. The scene encapsulates its deep yearning for freedom and the comfort of its natural habitat, juxtaposed with its current confinement and suffering.'

In [6]:
def process_dataset_in_batches(df, output_file=csv_path, batch_size=20, text_column='our_summary'):
    # 1. Ensure output column exists
    if "prompt_by_summary" not in df.columns:
        df["prompt_by_summary"] = ""

    total_rows = len(df)

    for start_idx in range(0, total_rows, batch_size):
        end_idx = min(start_idx + batch_size, total_rows)

        # Extract the batch (copy so we don't modify df by mistake)
        batch = df.iloc[start_idx:end_idx].copy()

        # 2. Skip rows that already have text in prompt_by_summary
        mask_needed = batch["prompt_by_summary"].astype(str).str.strip() == ""
        batch_to_process = batch[mask_needed]

        if batch_to_process.empty:
            print(f"Skipping rows {start_idx}–{end_idx-1}: already processed")
            continue

        # 3. Process only needed rows
        batch.loc[mask_needed, "prompt_by_summary"] = (
            batch_to_process[text_column].apply(generate_video_prompt)
        )

        # 4. Update original df for only processed rows
        df.loc[start_idx:end_idx-1, "prompt_by_summary"] = batch["prompt_by_summary"]

        print(f"Processed rows {batch_to_process.index.tolist()}")

    # 5. Save entire df only once at the end
    df.to_csv(output_file, index=False)
    print("All batches processed and saved!")

In [7]:
process_dataset_in_batches(df,batch_size=10)

Skipping rows 0–9: already processed
Skipping rows 10–19: already processed
Skipping rows 20–29: already processed
Skipping rows 30–39: already processed
Skipping rows 40–49: already processed
Skipping rows 50–59: already processed
Skipping rows 60–69: already processed
Skipping rows 70–79: already processed
Skipping rows 80–89: already processed
Skipping rows 90–99: already processed
Skipping rows 100–109: already processed
Skipping rows 110–119: already processed
Skipping rows 120–129: already processed
Skipping rows 130–139: already processed
Skipping rows 140–149: already processed
All batches processed and saved!
